## Historical Feature Ablations

I examined how much the older earnings events contributed to XGBoost's predictive performance. I compared the full four-event history against a version using only the most recent earnings event. I also tested whether removing the explicitly engineered change and trend features affected performance.

I used the same eligible observations and chronological validation folds from 2020 through 2023 for each configuration. The XGBoost hyperparameters remained unchanged and the reserved 2024 holdout was excluded.

In [11]:
import json
import pandas as pd

from IPython.display import display
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "ablations"

ablation_path = RESULTS_DIR / "xgboost_feature_ablations.json"

with ablation_path.open(encoding="utf-8") as file:
    ablation_results = json.load(file)


print(
    "Validation years:", [fold["validation_year"] for fold in ablation_results["folds"]]
)

print("\nFeature configurations:")

for name, columns in ablation_results["feature_sets"].items():
    print(f"{name}: {len(columns)} original features")

print(f"\nReserved holdout samples: {ablation_results['holdout_samples']:,}")
print(f"Holdout used: {ablation_results['holdout_used']}")

Validation years: [2020, 2021, 2022, 2023]

Feature configurations:
full_history: 25 original features
latest_event_only: 6 original features
without_trends: 22 original features

Reserved holdout samples: 7,996
Holdout used: False


### Feature Configurations

I compared three feature configurations to examine the contribution of older earnings information.

The Full History configuration used all 25 original features derived from the four preceding recorded earnings events.

For Latest Event Only, I kept the previous event's actual EPS, estimated EPS, surprise, beat indicator and report month and quarter. I removed the older lag features and the summary features calculated from multiple events, leaving six original features.

For Without Trends I retained the historical lag features but removed `eps_change1_2`, `eps_trend4` and `surprise_trend4`. This left 22 original features and allowed me to test whether the explicitly engineered change and trend features provided additional predictive value.

I kept the same observations, chronological validation folds and XGBoost hyperparameters across all configurations.

In [12]:
feature_sets = ablation_results["feature_sets"]

full_history_features = feature_sets["full_history"]
latest_event_features = feature_sets["latest_event_only"]
without_trends_features = feature_sets["without_trends"]

removed_trend_features = [
    feature
    for feature in full_history_features
    if feature not in without_trends_features
]

print("Latest-event-only features:")

for feature in latest_event_features:
    print(f"  {feature}")

print("\nFeatures removed in the without-trends configuration:")

for feature in removed_trend_features:
    print(f"  {feature}")

Latest-event-only features:
  actual_eps_lag1
  estimate_eps_lag1
  surprise_eps_lag1
  beat_lag1
  previous_report_month
  previous_report_quarter

Features removed in the without-trends configuration:
  eps_change1_2
  eps_trend4
  surprise_trend4


### Performance Comparison

I compared the average classification performance of the three feature configurations across the 2020–2023 validation years. I used macro F1 as the primary metric and also examined accuracy and ROC-AUC. Each configuration used the same observations and XGBoost settings which allowed me to examine how the choice of historical features affected performance.

In [13]:
configuration_names = {
    "full_history": "Full History",
    "latest_event_only": "Latest Event Only",
    "without_trends": "Without Trends",
}

performance_rows = []

for name, label in configuration_names.items():
    metrics = ablation_results["mean_performance"][name]

    performance_rows.append(
        {
            "Configuration": label,
            "Macro F1": metrics["macro_f1"],
            "Accuracy": metrics["accuracy"],
            "ROC-AUC": metrics["roc_auc"],
        }
    )

performance_table = pd.DataFrame(performance_rows)

display(performance_table.round(4))

,Configuration,Macro F1,Accuracy,ROC-AUC
0,Full History,0.6252,0.6689,0.6946
1,Latest Event Only,0.6133,0.6604,0.6759
2,Without Trends,0.6254,0.6689,0.6938


#### Interpretation

The Full History configuration performed better than Latest Event Only, with mean macro F1 increasing from 0.6133 to 0.6252. ROC-AUC also improved from 0.6759 to 0.6946. This suggests that the older earnings events contain useful information beyond the most recent result.

Removing the trend features had very little effect. Macro F1 increased slightly to 0.6254 while ROC-AUC decreased to 0.6938. These differences were small enough that I found no clear benefit from including the explicit change and trend features.

The underlying historical EPS values were still available in the Without Trends configuration, so XGBoost may have learned similar relationships directly from those inputs.

### Performance Across Validation Years

I examined the macro F1 scores for each validation year to see whether the differences between feature configurations were consistent over time. I compared the Full History configuration against Latest Event Only and Without Trends using the same chronological validation folds.

In [14]:
yearly_rows = []

for fold in ablation_results["folds"]:
    models = fold["models"]

    yearly_rows.append(
        {
            "Year": fold["validation_year"],
            "Full History": models["full_history"]["macro_f1"],
            "Latest Event Only": models["latest_event_only"]["macro_f1"],
            "Without Trends": models["without_trends"]["macro_f1"],
        }
    )

yearly_table = pd.DataFrame(yearly_rows)

display(yearly_table.round(4))

,Year,Full History,Latest Event Only,Without Trends
0,2020,0.6070,0.5976,0.6079
1,2021,0.6582,0.6443,0.6573
2,2022,0.6210,0.6126,0.6216
3,2023,0.6146,0.5988,0.6148


#### Interpretation

Full History achieved a higher macro F1 than Latest Event Only in every validation year. The largest difference occurred in 2023 where macro F1 increased from 0.5988 to 0.6146. The smallest difference was in 2022 at approximately 0.0084.

I found that including older earnings information consistently improved XGBoost's performance across the four validation folds although the size of the improvement varied.

Removing the trend features had almost no effect in any year. Without Trends performed slightly better in three of the four folds, but the differences were all smaller than 0.001. This provides little evidence that the engineered trend features improved classification performance in this experiment.

### Conclusion

I compared XGBoost using the full four-event history against a version using only the most recent recorded earnings event. Full History achieved higher macro F1 in every validation year with an average of 0.6252 compared with 0.6133, suggesting that older earnings events provide useful predictive information beyond the most recent result.

Removing the explicitly engineered change and trend features had little effect on performance. The historical EPS values were still available so XGBoost may have learned similar relationships directly from those inputs. These results support using the full earnings history for the final evaluation, although I found little benefit from the additional trend features. I will consider these findings when selecting the final model configuration for the reserved 2024 holdout.